# Colab GPU runtime for Phillips_UC2

Run this notebook **in Google Colab** (Runtime > Change runtime type > GPU). It sets up an SSH tunnel so you can attach VS Code's **Remote - SSH** extension directly to this Colab VM and run/debug the repo on its GPU as if it were local.

Steps: run all cells below in order, then follow the printed VS Code instructions.

In [33]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [34]:
# One-time setup: SSH server + cloudflared tunnel (no ngrok account needed, no flaky colab_ssh package)
!apt-get -qq -y install openssh-server > /dev/null
!mkdir -p /var/run/sshd
!echo "root:changeme" | chpasswd
!sed -i "s/#PermitRootLogin.*/PermitRootLogin yes/" /etc/ssh/sshd_config
!sed -i "s/#PasswordAuthentication.*/PasswordAuthentication yes/" /etc/ssh/sshd_config
!service ssh restart

!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared


 * Restarting OpenBSD Secure Shell server sshd
   ...done.
/content/cloudflared: Text file busy


In [35]:
import subprocess, time, re

# start tunnel in background, log to file so we can grab the printed URL
proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "ssh://localhost:22", "--logfile", "/content/cloudflared.log"]
)

url = None
for _ in range(30):
    time.sleep(1)
    try:
        log = open("/content/cloudflared.log").read()
    except FileNotFoundError:
        continue
    match = re.search(r"https://[a-zA-Z0-9.-]+trycloudflare\.com", log)
    if match:
        url = match.group(0)
        break

if not url:
    raise RuntimeError("cloudflared didn't print a URL in time -- check /content/cloudflared.log")

hostname = url.replace("https://", "")
print(f"""Add this to your local ~/.ssh/config, then VS Code Remote-SSH -> Connect to Host -> colab
(password: changeme)

Host colab
    HostName {hostname}
    User root
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h
""")


Add this to your local ~/.ssh/config, then VS Code Remote-SSH -> Connect to Host -> colab
(password: changeme)

Host colab
    HostName someone-dancing-improving-wireless.trycloudflare.com
    User root
    Port 22
    ProxyCommand cloudflared access ssh --hostname %h



The cell above prints an SSH `Host` block. It requires `cloudflared` installed **locally** too (the `ProxyCommand` runs it on your machine) — install it from https://github.com/cloudflare/cloudflared/releases or `winget install --id Cloudflare.cloudflared`.

Paste the block into your local `~/.ssh/config` (VS Code Command Palette > "Remote-SSH: Open SSH Configuration File"), save, then Command Palette > "Remote-SSH: Connect to Host" > `colab`. Enter password `changeme` when prompted.

In [36]:
# Get the repo onto the Colab VM
!git clone https://github.com/sormazabal/Phillips_UC2.git /content/Phillips_UC2


fatal: destination path '/content/Phillips_UC2' already exists and is not an empty directory.


In [37]:
%cd /content/Phillips_UC2
!pip install -q -r requirements.txt

/content/Phillips_UC2


From the VS Code window connected via Remote-SSH, open `/content/Phillips_UC2` as the workspace folder, select the Python interpreter, and run e.g.:

```bash
python scripts/train.py --config config.yaml
```

Keep this Colab tab open -- closing it kills the tunnel and the VM.

In [39]:
%env PYTHONPATH=/content/Phillips_UC2
!python scripts/train.py --config config.yaml

env: PYTHONPATH=/content/Phillips_UC2
Traceback (most recent call last):
  File "/content/Phillips_UC2/scripts/train.py", line 88, in <module>
    main()
  File "/content/Phillips_UC2/scripts/train.py", line 25, in main
    train_dataset = ArcadeDataset(
                    ^^^^^^^^^^^^^^
  File "/content/Phillips_UC2/src/data/dataset.py", line 74, in __init__
    with open(syntax_ann_file, "r") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: './Arcade/syntax/train/annotations/train.json'


In [43]:
!scp -r Arcade colab:/content/Phillips_UC2/Arcade

ssh: Could not resolve hostname colab: Name or service not known
lost connection


In [40]:
!ls -la /content/Phillips_UC2/Arcade/syntax/train/annotations/

ls: cannot access '/content/Phillips_UC2/Arcade/syntax/train/annotations/': No such file or directory
